In [1]:
# Election cycle
years="2026-2027"

# List of candidates in the order listed on the voting site., which in turn was the order in which
# candidates nominated themselves.
candidates = [
    "Pavan Kumar Reddy Vaddamani (Tata Consultancy Services)",
    "Deanne Harwood (DHS/CISA)",
    "Suraj Panker (Depronto Infotech)",
    "Ganesan Ramalingam (Microsoft)",
    "Alexandre Eichenberger (IBM)",
    "Brian Nguyen (NVIDIA)",
    "Andreas Fehlner (TRUMPF Laser SE)",
    "Yuanyuan Chen (Innoxsea)", 
    "Christian Bourjau (QuantCo)",
    "Pavan Kumar Upadhyayula (OneTrust)",
    "Odysseas Chatzopoulos (National and Kapodistrian University of Athens)",
    "Javier Martinez (Intel)",
]

# 4 letter identifier to enable simple identification in verbose results.
candidate_short = ["PavV","Dean","Sura","Rama","Alex","Bria","Andr","Yuan","Chri","PavU","Odys","Javi"]

# Votes with company (added manually), 
#   name (replaced manually from email address, which should be avoided in published docs), 
#   and votes ranking for each candidate.
# Raw info cannot be pulled from the voting site as cvs file as order is randomized and no email identifier.
# Raw info is extracted from a saved HTML file into a text file (imported without formatting), and then 
# edited to have one ballot per line, with company and name instead of email address. Numerical values
# corresponds to the candidates in the same order as listed in candidates and candidate_short arrays.
votes = [
    ["eSOL Co","Takeshi Watanabe",11,7,10,3,4,1,5,8,6,12,9,2],
    ["IBM","Adrian Selk",12,12,12,12,1,12,12,12,12,12,12,12],
    ["Intel","Agnes So",9,11,12,3,4,5,3,8,4,7,8,1],
    ["independent","Yash Solanki",5,7,4,1,3,9,8,6,2,10,12,11],
    ["IRT Saint Exupery","Eric JENN",12,12,12,12,1,4,1,12,1,12,12,5],
    ["Multicoreware Inc","Jagadeesh V",5,8,7,4,1,3,6,12,10,10,11,2],
    ["Microsoft","Justin Chu",12,12,12,1,2,2,2,12,9,12,12,2],
    ["IBM","Haruki Imai",12,12,12,12,1,12,12,12,12,12,12,12],
    ["IBM","Alexandre Eichenberger",3,3,3,1,1,1,1,3,2,3,3,1],
    ["ONERA","Jean-Loup FARGES",12,7,11,3,2,9,5,6,10,4,1,8],
    ["NVIDIA","Yuan Yao",12,12,12,2,3,1,4,12,5,12,12,12],
    ["Multicoreware Inc","Logeshwaran Elanchelian",7,7,12,12,1,12,7,8,7,12,7,7],
    ["IBM","Tong Chen",12,12,12,12,1,12,12,12,12,12,12,12],
    ["Intel","Javier Martinez",12,12,12,3,4,5,2,12,12,12,12,1],
    ["IBM","Christopher Munoz",12,12,12,12,1,12,2,12,12,12,12,12],
    ["Nanyang Technological University","Yuanyuan Chen",9,6,11,5,8,7,2,1,3,12,10,4],
    ["AMD","Jonas Rickert",12,12,12,1,1,12,12,12,12,12,12,12],
    ["NVIDIA","Kevin Chen",12,12,12,2,3,1,12,12,12,12,12,12],
    ["Microsoft","G. Ramalingam",12,12,12,1,2,3,2,12,3,12,12,3],
    ["NVIDIA","Mayank Kaushik",12,12,12,3,4,1,2,12,12,12,12,5],
    ["M&T Bank","Brian Parbhu",8,7,12,5,3,4,1,11,9,10,2,6],
    ["emmtrix","Timo Stripf",12,12,12,12,12,12,1,12,12,12,12,12],
    ["TelePIX","Hyun Gyu Kim",12,3,9,5,1,6,8,11,10,4,2,7],
    ["Intel","Yamini Nimmagadda",5,6,6,1,2,4,3,5,4,6,6,1],
    ["Quantco","Christian Bourjau",8,12,12,5,4,6,2,3,1,12,12,7],
    ["TRUMPF Laser SE","Andreas Fehlne",12,12,12,3,2,7,1,5,4,12,8,6],
    ["IBM","Tung Le",12,12,12,2,1,3,5,12,12,12,12,4],
]


In [2]:
import numpy as np

# From the list of all ballots, seclect the ones for this company (or all if company is "ALL").
def ballot_for_company(ballots, select_company, verbose=0):
    res = []
    for b in ballots:
        curr_company, curr_ballot = b[0], b[2:]
        if select_company == 'ALL' or curr_company == select_company:
            res.append(curr_ballot)
            if verbose:
                print(curr_company, curr_ballot)
    return np.array(res)

# Help for debugging.
def justified_numbers(a):
    return map(lambda n: str(n).rjust(4), a)

def print_justified_numbers(title, a):
    print(title)
    print("  ", *candidate_short)
    print("  ", *justified_numbers(a))

# Create a new ballot reflecting the choices in the given ballot sample. 
# Verbose of 1 print debugging info, verbose of 2 additionally print every ballot used.
def rank(ballots, candidates, verbose=0):
    cnum = candidates.size
    num_ballots = 0
    #m[a,b] is "time folks preferred candidate b over a".
    m = np.zeros((cnum, cnum), dtype=int)
    for b in ballots:
        num_ballots += 1
        for j in range(cnum):
            for k in range(cnum):
                if int(b[k])>int(b[j]):
                    m[k][j] = m[k][j] + 1
    total = np.sum(m, axis=0) # sum over columns
    # (candidate id, preference score) ranked by most preferred first
    rank = sorted(zip(range(cnum),total), reverse=True, key=lambda record: record[1])
    rank_named = sorted(zip(candidate_short,total), reverse=True, key=lambda record: record[1])
    # New ballot: where candidates that have the same preferrence have the same score (low is better).
    new_ballot = np.zeros(cnum, dtype=int)
    preference = 0
    previous_score = 10000000
    for p in range(cnum): # By order of decreasing popularity.
        curr_candidate_id, curr_candidate_score = rank[p]
        if curr_candidate_score != previous_score:
            preference +=1 # Different score, advance by 1.
        new_ballot[curr_candidate_id] = preference
        previous_score = curr_candidate_score   
    if verbose:
        print("##########################################################")
        print("Tabulating", num_ballots, "votes.")
        if verbose > 1:
            i = 1
            for b in ballots:
                print("  ", i, ":", b)
                i += 1
        print("Matrix m[a, b], times voters preferred candidate b over a:")
        print("          ", *candidate_short, sep = " ")
        for i in range(cnum):
            m_i_str = justified_numbers(m[i])
            print("  ", i, candidate_short[i], ":", *m_i_str, sep = " ")
        print_justified_numbers("Totals (sum over columns of m)", total)
        print_justified_numbers("Rank (candidate id, candidate preference score)",rank_named)
        print_justified_numbers("New ballot", new_ballot)
    return (num_ballots, new_ballot)

# Create a ballot that reflects all the votes per company.
# Verbose of 1 print debugging info, verbose of 2 additionally print every ballot used.
def rank_per_company(votes, candidates, verbose=0):
    companies = np.unique(votes[:, 0])
    ballots_per_company = []
    ballots_per_company_with_name = []
    (tot_num_ballots, counted_num_ballot) = (np.shape(votes)[0], 0)
    print("Participating companies:", companies, "\n")
    for com in companies:
        print("Ballot for company = ", com)
        (company_num_ballots, new_ballot) = rank(ballot_for_company(votes, com), candidates, verbose=verbose)
        counted_num_ballot += company_num_ballots
        print("  is ", new_ballot, " with ", company_num_ballots, "counted ballots\n")
        ballots_per_company.append(new_ballot)
        ballots_per_company_with_name.append([com, new_ballot])
    assert counted_num_ballot == tot_num_ballots
    print("Final ballot with one vote per company")
    (final_num_ballot, final_ballot) = rank(np.array(ballots_per_company), candidates, verbose=verbose)
    print("  is ", final_ballot, " with ", final_num_ballot, "counted ballots\n")
    assert final_num_ballot == np.shape(companies)[0]
    if verbose:
        print("Final ballot = ", final_ballot)
        print("")
    return (counted_num_ballot, final_ballot, ballots_per_company_with_name)

# Translate a ballot into the list of ordered candidates using names.
def ballot_to_candidates(ballot, candidates):
    cnum = candidates.size
    # (candidate id, order) ranked by most preferred first
    rank = sorted(zip(range(cnum), ballot), key=lambda record: record[1])
    for r in rank:
        num = str(r[1])+". "
        print(num, candidates[r[0]])

In [3]:
# Use numpy arrays.
vv = np.array(votes)
cc = np.array(candidates)
verbose = 1 # 0 none, 1 some, 2 print ballots also.
#ballot_for_company(vv, 'INTEL')
#rank(ballot_for_company(vv, "INTEL"), cc, verbose=1)

print("Non-official vote where each voter get one voice. This is not the offical result and is only provided to investgate the difference between individual votes vs votes per company\n")
(total_votes1, ballot_per_voter) = rank(ballot_for_company(vv, "ALL"), cc, verbose=verbose)
print("\nBallot per voter =", ballot_per_voter)
ballot_to_candidates(ballot_per_voter, cc)
print("\n\n")

print("Official vote per our election rules, where votes are tallied by company first, and the final round includes exactly one ballot per company\n")
(total_votes2, final_ballot_per_company, ballots_per_company) = rank_per_company(vv, cc, verbose=verbose)
assert total_votes1 == total_votes2
print("\nFinal ballot (summarizing per company)=", final_ballot_per_company)
ballot_to_candidates(final_ballot_per_company, cc)



Non-official vote where each voter get one voice. This is not the offical result and is only provided to investgate the difference between individual votes vs votes per company

##########################################################
Tabulating 27 votes.
Matrix m[a, b], times voters preferred candidate b over a:
           PavV Dean Sura Rama Alex Bria Andr Yuan Chri PavU Odys Javi
   0 PavV :    0    5    4   20   26   19   19    7   14    3    6   17
   1 Dean :    5    0    2   19   25   16   19    7   12    2    5   15
   2 Sura :    7    7    0   20   26   19   21    9   15    4    8   18
   3 Rama :    1    2    0    0   13    7   10    3    4    1    4    7
   4 Alex :    0    1    0   11    0    5    8    2    3    0    2    5
   5 Bria :    2    5    1   13   19    0   14    6    7    2    4    8
   6 Andr :    2    2    1   12   15    7    0    2    2    2    2    7
   7 Yuan :    4    5    3   18   24   15   20    0   15    5    5   14
   8 Chri :    2    4    2   18   21

In [4]:
def printResultMD(years, ballots, aggregated_ballots, candidates, candidate_shortname, results, verbose=0):
    cnum = candidates.size
    # Table of votes
    print("# Vote results for the ONNX Steering Committee for " + years + ".")
    print("## Votes casted by Contributors.")
    str = "| Contributors | "
    for c in candidates:
        str += c + " | "
    print(str)
    str = "|"
    for c in range(cnum+1):
        str += "-------|"
    print(str)
    for b in ballots:
        str = "| " + b[1] + " (" + b[0] + ") | "
        for c in range(cnum):
            str += b[c+2] + " | "
        print(str)
    # Print ballots per company
    print("\n## Precedence per company.")
    print("| Member company | Precedence |")
    print("|----------------|-------------------------------------------------------------|")
    for b in aggregated_ballots:
        bb = np.array(b[1])
        rank = sorted(zip(candidate_shortname,bb), key=lambda record: record[1])
        if (verbose):
            print("ballot:", b)
            print("rank ", rank)
        str = "| " + b[0] + " | "
        for c in range(cnum):
            if c>0:
                if rank[c][1] == rank[c-1][1]:
                    str += " = "
                else:
                    str += " > "
            str += rank[c][0]
        print(str+ " |")        
    # Print final results
    print("\n## Final results (Schultz method, top 5 are elected).")
    ballot_to_candidates(results, candidates)

In [10]:
# Copy output in a separate file for easy consumption of the results.
printResultMD(years, vv, ballots_per_company, cc, candidate_short, final_ballot_per_company)

# Vote results for the ONNX Steering Committee for 2026-2027.
## Votes casted by Contributors.
| Contributors | Pavan Kumar Reddy Vaddamani (Tata Consultancy Services) | Deanne Harwood (DHS/CISA) | Suraj Panker (Depronto Infotech) | Ganesan Ramalingam (Microsoft) | Alexandre Eichenberger (IBM) | Brian Nguyen (NVIDIA) | Andreas Fehlner (TRUMPF Laser SE) | Yuanyuan Chen (Innoxsea) | Christian Bourjau (QuantCo) | Pavan Kumar Upadhyayula (OneTrust) | Odysseas Chatzopoulos (National and Kapodistrian University of Athens) | Javier Martinez (Intel) | 
|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|-------|
| Takeshi Watanabe (eSOL Co) | 11 | 7 | 10 | 3 | 4 | 1 | 5 | 8 | 6 | 12 | 9 | 2 | 
| Adrian Selk (IBM) | 12 | 12 | 12 | 12 | 1 | 12 | 12 | 12 | 12 | 12 | 12 | 12 | 
| Agnes So (Intel) | 9 | 11 | 12 | 3 | 4 | 5 | 3 | 8 | 4 | 7 | 8 | 1 | 
| Yash Solanki (independent) | 5 | 7 | 4 | 1 | 3 | 9 | 8 | 6 | 2 | 10 | 12 | 11 | 
| Eric JENN (IRT Saint 